<a href="https://colab.research.google.com/github/maggiecrowner/DS5001-Final-Project/blob/main/ParsedandAnnotatedData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parsed and Annotated Data

In [18]:
! git clone https://github.com/maggiecrowner/DS5001-Final-Project.git

fatal: destination path 'DS5001-Final-Project' already exists and is not an empty directory.


In [19]:
import pandas as pd
import glob
import os

path = '/content/DS5001-Final-Project/data/Source Data'
# read in only 10 of the csv files, to meet row requirements
for_corpus = [
    "PostMalone.csv",
    "TaylorSwift.csv",
    "EdSheeran.csv",
    "JustinBieber.csv",
    "ColdPlay.csv",
    "Maroon5.csv",
    "Beyonce.csv",
    "LadyGaga.csv",
    "BillieEilish.csv",
    "ArianaGrande.csv"
]
all_csvs = glob.glob(os.path.join(path, '**/*.csv'), recursive=True)
csv_files = [f for f in all_csvs if os.path.basename(f) in for_corpus]

print(f"Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f"  {os.path.relpath(f, path)}")

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    df['_source_file'] = os.path.basename(f)
    dfs.append(df)

source_data = pd.concat(dfs, ignore_index=True)
print(f"\n Corpus shape: {source_data.shape}")
source_data.head()

Found 10 CSV files:
  ArianaGrande.csv
  JustinBieber.csv
  LadyGaga.csv
  ColdPlay.csv
  TaylorSwift.csv
  Maroon5.csv
  Beyonce.csv
  EdSheeran.csv
  PostMalone.csv
  BillieEilish.csv

 Corpus shape: (3073, 8)


,Artist,Title,Album,Date,Lyric,Year,_source_file,Unnamed: 0
0,Ariana Grande,"​thank u, next","thank u, next",2018-11-03,thought i'd end up with sean but he wasn't a m...,2018.0,ArianaGrande.csv,NaN
1,Ariana Grande,7 rings,"thank u, next",2019-01-18,yeah breakfast at tiffany's and bottles of bub...,2019.0,ArianaGrande.csv,NaN
2,Ariana Grande,​God is a woman,Sweetener,2018-07-13,you you love it how i move you you love it how...,2018.0,ArianaGrande.csv,NaN
3,Ariana Grande,Side To Side,Dangerous Woman,2016-05-20,ariana grande nicki minaj i've been here all ...,2016.0,ArianaGrande.csv,NaN
4,Ariana Grande,​​no tears left to cry,Sweetener,2018-04-20,right now i'm in a state of mind i wanna be in...,2018.0,ArianaGrande.csv,NaN


## LIB

In [20]:
source_data['track_id'] = source_data['Artist'] + ' — ' + source_data['Title'] + ' (' + source_data['Album'] + ')'

# add additional columns for model summarization
source_data['doc_length_words'] = source_data['Lyric'].astype(str).str.split().str.len()
source_data['doc_length_chars'] = source_data['Lyric'].astype(str).str.len()
source_data['Year'] = pd.to_numeric(source_data['Year'], errors='coerce')
source_data['Decade'] = (source_data['Year'].dropna().astype(int) // 10 * 10).astype(str) + 's'

LIB = (source_data[['track_id', 'Artist', 'Album', 'Title', 'Year', 'Decade',
                     'doc_length_words', 'doc_length_chars']]
       .drop_duplicates()
       .set_index('track_id'))

print(f" LIB shape: {LIB.shape}")
LIB.head()

 LIB shape: (3073, 7)


,Artist,Album,Title,Year,Decade,doc_length_words,doc_length_chars
track_id,,,,,,,
"Ariana Grande — ​thank u, next (thank u, next)",Ariana Grande,"thank u, next","​thank u, next",2018.0,2010s,463,2305
"Ariana Grande — 7 rings (thank u, next)",Ariana Grande,"thank u, next",7 rings,2019.0,2010s,490,2184
Ariana Grande — ​God is a woman (Sweetener),Ariana Grande,Sweetener,​God is a woman,2018.0,2010s,439,1975
Ariana Grande — Side To Side (Dangerous Woman),Ariana Grande,Dangerous Woman,Side To Side,2016.0,2010s,551,2727
Ariana Grande — ​​no tears left to cry (Sweetener),Ariana Grande,Sweetener,​​no tears left to cry,2018.0,2010s,500,2307


In [21]:
LIB.to_csv('/content/DS5001-Final-Project/data/LIB.csv', sep='|', index=True)

## CORPUS

In [22]:
import nltk
import re
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt')
nltk.download('punkt_tab')

pos_group_map = {
    'NN': 'NOUN', 'NNS': 'NOUN', 'NNP': 'NOUN', 'NNPS': 'NOUN',
    'VB': 'VERB', 'VBD': 'VERB', 'VBG': 'VERB', 'VBN': 'VERB', 'VBP': 'VERB', 'VBZ': 'VERB',
    'JJ': 'ADJ',  'JJR': 'ADJ',  'JJS': 'ADJ',
    'RB': 'ADV',  'RBR': 'ADV',  'RBS': 'ADV',
}

records = []
for _, row in source_data.iterrows():
    artist = row['Artist']
    album  = row['Album']
    track  = row['Title']
    lyric  = str(row['Lyric'])

    words    = lyric.split()
    pos_tags = nltk.pos_tag(words)

    for word_id, (token_str, pos) in enumerate(pos_tags):
        term_str  = re.sub(r'[^\w\s]', '', token_str).lower()
        pos_group = pos_group_map.get(pos, 'OTHER')

        records.append({
            'Artist':    artist,
            'Album':     album,
            'Title':     track,
            'WordID':    word_id,
            'token_str': token_str,
            'term_str':  term_str,
            'pos':       pos,
            'pos_group': pos_group,
        })

CORPUS = pd.DataFrame(records)
CORPUS = CORPUS.set_index(['Artist', 'Album', 'Title', 'WordID'])
print(f" Corpus shape: {CORPUS.shape}")
CORPUS.head()

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


 Corpus shape: (987386, 4)


token_str term_str  pos  \
Artist        Album         Title          WordID                           
Ariana Grande thank u, next ​thank u, next 0        thought  thought   NN   
                                           1            i'd       id  NNS   
                                           2            end      end  VBP   
                                           3             up       up   RP   
                                           4           with     with   IN   

                                                  pos_group  
Artist        Album         Title          WordID            
Ariana Grande thank u, next ​thank u, next 0           NOUN  
                                           1           NOUN  
                                           2           VERB  
                                           3          OTHER  
                                           4          OTHER

In [23]:
CORPUS.to_csv('/content/DS5001-Final-Project/data/CORPUS.csv', sep='|', index=True)

## VOCAB

In [24]:
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import numpy as np

ps = PorterStemmer()

n = CORPUS['term_str'].value_counts().rename('n')
doc_term = CORPUS.reset_index().groupby('Title')['term_str'].apply(set)
N_docs = len(doc_term)

df_counts = (CORPUS.reset_index()
             .groupby('term_str')['Title']
             .nunique()
             .rename('df'))

VOCAB = pd.DataFrame({'n': n, 'df': df_counts})
VOCAB.index.name = 'term_str'

VOCAB['p']            = VOCAB['df'] / N_docs
VOCAB['i']            = np.log2(N_docs / VOCAB['df'])
VOCAB['dfidf']        = VOCAB['df'] * VOCAB['i']
VOCAB['porter_stem']  = VOCAB.index.map(lambda t: ps.stem(t))
VOCAB['max_pos']      = CORPUS.groupby('term_str')['pos'].agg(lambda x: x.value_counts().idxmax())
VOCAB['max_pos_group']= CORPUS.groupby('term_str')['pos_group'].agg(lambda x: x.value_counts().idxmax())
VOCAB['stop']         = VOCAB.index.map(lambda t: int(t in ENGLISH_STOP_WORDS))
VOCAB['ngram_length'] = VOCAB.index.map(lambda t: len(t.split()))
VOCAB = VOCAB.drop(columns=['df'])

print(f" VOCAB shape: {VOCAB.shape}")
VOCAB.head()

 VOCAB shape: (19104, 9)


,n,p,i,dfidf,porter_stem,max_pos,max_pos_group,stop,ngram_length
term_str,,,,,,,,,
,1,0.000333,11.553629,11.553629,,'',OTHER,0,0
0,151,0.029275,5.094198,448.289395,0,CD,OTHER,0,1
00,33,0.008317,6.909773,172.744328,00,CD,OTHER,0,1
000,14,0.001996,8.968667,53.812001,000,CD,OTHER,0,1
000s,1,0.000333,11.553629,11.553629,000,CD,OTHER,0,1


In [25]:
VOCAB.to_csv('/content/DS5001-Final-Project/data/VOCAB.csv', sep='|', index=True)

# EDA / Summary statistics of the tables

In [26]:
print("Average length of document, in characters:", source_data['Lyric'].str.len().mean())

Average length of document, in characters: 1585.400655737705


In [27]:
print("Number of observations/tokens:", len(CORPUS['token_str']))

Number of observations/tokens: 987386


In [28]:
print("Number of observations/vocab:", len(VOCAB['dfidf']))

Number of observations/vocab: 19104


In [29]:
top20 = (VOCAB[VOCAB['stop'] == 0]
         .sort_values('dfidf', ascending=False)
         .head(20))
print("Top 20 significant words by DFIDF:")
top20[['n', 'p', 'i', 'dfidf', 'porter_stem', 'max_pos_group']]

Top 20 significant words by DFIDF:


,n,p,i,dfidf,porter_stem,max_pos_group
term_str,,,,,,
baby,5378,0.366600,1.447721,1595.388305,babi,NOUN
time,2992,0.371590,1.428216,1595.317075,time,NOUN
yeah,6556,0.373586,1.420487,1595.206993,yeah,NOUN
say,3023,0.355955,1.490234,1594.550608,say,VERB
youre,4001,0.387558,1.367515,1593.155038,your,NOUN
got,4441,0.390552,1.356413,1592.428393,got,VERB
want,4758,0.337658,1.566365,1589.860761,want,VERB
pre,2130,0.335329,1.576349,1588.960165,pre,NOUN
make,2513,0.313041,1.675578,1576.719257,make,VERB
